# Portfolio context

Original STAT 303-2 course analysis by Ben Saperstein, Winter 2026. Analytical cells and saved outputs are preserved below. Read [validation notes](../docs/methods-and-limitations.md) before interpreting model scores: turning speed contains the target, and a corrected evaluation has not been run. The original export is not included. The portfolio copy uses a portable Python kernel selector.


In [52]:
#| echo: false

# =========================
# cstimer session36 → tidy DataFrame
# =========================

import json, re, numpy as np, pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

# More tools here
from sklearn.linear_model import Ridge, Lasso, LinearRegression, LogisticRegression, RidgeCV, LassoCV, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, accuracy_score, recall_score, roc_curve, auc, roc_auc_score, r2_score
from sklearn.model_selection import cross_val_score, cross_val_predict, cross_validate, train_test_split

import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

import warnings
warnings.filterwarnings("ignore")

# ---- 1) Load JSON ----
filename = "cstimer_data.txt"   # <- change if needed
with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

if "session36" not in data:
    raise KeyError(f"'session36' not found. Available keys (truncated): {list(data.keys())[:10]} ...")

session_data = data["session36"]  # list of solves

# ---- 2) Build base rows ----
rows = []
for idx, entry in enumerate(session_data):
    time_ms = scramble = timestamp = None
    events = []

    if isinstance(entry, list):
        # Typical: [ [0, time_ms], scramble, comment, timestamp, events? ]
        if len(entry) >= 1 and isinstance(entry[0], list) and len(entry[0]) >= 2:
            time_ms = entry[0][1]
        if len(entry) >= 2:
            scramble = entry[1]
        if len(entry) >= 4:
            timestamp = entry[3]
        if len(entry) >= 5 and isinstance(entry[4], list):
            events = entry[4]
    elif isinstance(entry, dict):
        time_ms   = entry.get("time_ms") or entry.get("time")
        scramble  = entry.get("scramble")
        timestamp = entry.get("timestamp") or entry.get("date")
        events    = entry.get("events", []) or []

    rows.append({
        "solve_index": idx,
        "time_ms": time_ms,
        "scramble": scramble,
        "timestamp": timestamp,
        "events": events
    })

solve_data = pd.DataFrame(rows)

# ---- 3) Date/Time split + weekday & time-of-day bucket ----
solve_data["datetime"] = pd.to_datetime(solve_data["timestamp"], unit="s", errors="coerce")
solve_data["date"] = solve_data["datetime"].dt.date
solve_data["time"] = solve_data["datetime"].dt.time
solve_data["day_of_week"] = solve_data["datetime"].dt.day_name()

def _bucket(hour: int) -> str:
    if 6 <= hour < 12:  return "Morning"
    if 12 <= hour < 18: return "Afternoon"
    if 18 <= hour < 24: return "Evening"
    return "Late Night"

solve_data["time_of_day_bucket"] = solve_data["datetime"].dt.hour.apply(lambda h: _bucket(int(h)) if pd.notna(h) else "Unknown")

# ---- 4) Move parsing & metrics ----
rot_pat  = re.compile(r"^\s*(x|y|z)(2|')?\s*$", re.IGNORECASE)
turn_pat = re.compile(
    r"^\s*(?:"
    r"(?:[RLUDFB](?:w)?|[MES])"      # face/wide/slice
    r"|2-2[RLUDFB]w"                 # multi-layer e.g., 2-2Lw
    r")(?:2|')?\s*$",
    re.IGNORECASE
)
move_rec_pat = re.compile(r"^\s*([^@]+)@(\d+)\s*$")
turn_token_pat = re.compile(
    r"^\s*((?:[RLUDFB](?:w)?|[MES]|2-2[RLUDFB]w))(2|')?\s*$",
    re.IGNORECASE
)

def parse_move_list(ev_list):
    """
    Robustly parse events that may come as ["R@100 U@200 ..."] (one long string)
    or as multiple strings. Returns:
      moves: [(token, t_ms)] sorted by time
      turns_ts: [t_ms] for turns
      rots_ts:  [t_ms] for rotations
      turn_tokens_in_order: [token] for turn moves in chronological order
    """
    if not isinstance(ev_list, list) or len(ev_list) == 0:
        return [], [], [], []

    # Flatten: split any string item into whitespace-separated tokens
    raw_tokens = []
    for e in ev_list:
        if isinstance(e, str):
            raw_tokens.extend([p for p in e.strip().split() if p])

    parsed = []
    for tok in raw_tokens:
        m = move_rec_pat.match(tok)
        if not m:
            continue
        move_token = m.group(1).strip()
        t = int(m.group(2))
        parsed.append((move_token, t))

    parsed.sort(key=lambda x: x[1])

    turns_ts = [t for tok, t in parsed if turn_pat.match(tok)]
    rots_ts  = [t for tok, t in parsed if rot_pat.match(tok)]
    turn_tokens_in_order = [tok for tok, _ in parsed if turn_pat.match(tok)]

    return parsed, turns_ts, rots_ts, turn_tokens_in_order

def inverse_turn_token(tok: str):
    """
    Inverse for turns (not rotations):
      R <-> R' , R2 -> R2 , Rw <-> Rw' , 2-2Lw <-> 2-2Lw'
    """
    if not isinstance(tok, str):
        return None
    m = turn_token_pat.match(tok.strip())
    if not m:
        return None
    base, mod = m.group(1), (m.group(2) or "")
    if mod == "2":
        inv_mod = "2"
    elif mod == "'":
        inv_mod = ""
    else:
        inv_mod = "'"
    return f"{base}{inv_mod}"

def longest_gap_ms(times_ms):
    if len(times_ms) < 2:
        return 0
    diffs = np.diff(times_ms)
    return int(diffs.max()) if len(diffs) else 0

def fastest_k_window_ms(times_ms, k=10):
    if len(times_ms) < k:
        return None
    best = None
    for i in range(len(times_ms) - k + 1):
        w = times_ms[i + k - 1] - times_ms[i]
        if best is None or w < best:
            best = w
    return int(best)

def per_solve_metrics(row):
    time_ms = row["time_ms"]
    events  = row["events"]

    moves, turns_ts, rots_ts, turn_tokens = parse_move_list(events)

    last_evt_ms = moves[-1][1] if moves else None
    if isinstance(time_ms, (int, float, np.integer, np.floating)) and time_ms and time_ms > 0:
        dur_ms = int(time_ms)
    else:
        dur_ms = last_evt_ms

    # Counts
    number_of_turns     = int(len(turns_ts))
    number_of_rotations = int(len(rots_ts))
    total_move_count    = int(len(moves))

    # Speed & pauses
    turn_speed_tps = (number_of_turns / (dur_ms / 1000.0)) if (dur_ms and dur_ms > 0) else np.nan
    longest_pause = longest_gap_ms([t for _, t in moves])
    fastest10_ms  = fastest_k_window_ms(turns_ts, k=10)

    # Scramble length
    scramble = row["scramble"]
    scramble_len = int(len(str(scramble).split())) if pd.notna(scramble) else 0

    # Wasted moves (turns only): immediate inverse pairs
    wasted = 0
    for i in range(len(turn_tokens) - 1):
        inv = inverse_turn_token(turn_tokens[i])
        if inv is not None and turn_tokens[i + 1] == inv:
            wasted += 1

    return pd.Series({
        "solve_time_s": (dur_ms / 1000.0) if dur_ms is not None else np.nan,   # float
        "number_of_turns": number_of_turns,                                     # int
        "turn_speed_tps": float(turn_speed_tps) if turn_speed_tps == turn_speed_tps else np.nan,
        "number_of_rotations": number_of_rotations,                             # int
        "total_move_count": total_move_count,                                   # int
        "longest_pause_ms": int(longest_pause),                                 # int
        "fastest_burst_10_moves_ms": fastest10_ms,                              # nullable int
        "wasted_moves": int(wasted),                                            # int
        "scramble_length": scramble_len                                         # int
    })

solve_data = pd.concat([solve_data, solve_data.apply(per_solve_metrics, axis=1)], axis=1)

# ---- 5) Final column order (day_of_week after date; then time; then bucket) ----
final_cols = [
    "solve_index",
    "solve_time_s",
    "date",
    "day_of_week",
    "time",
    "time_of_day_bucket",
    "number_of_turns",
    "turn_speed_tps",
    "number_of_rotations",
    "total_move_count",
    "longest_pause_ms",
    "fastest_burst_10_moves_ms",
    "wasted_moves",
    "scramble",
    "scramble_length",
]

solve_data_final = solve_data[final_cols].copy()

# ---- 6) Convert integers where possible; keep nullable Int64 for fastest_burst_10_moves_ms ----
int_cols_strict = [
    "solve_index", "number_of_turns", "number_of_rotations",
    "total_move_count", "longest_pause_ms", "wasted_moves", "scramble_length"
]
for c in int_cols_strict:
    solve_data_final[c] = pd.to_numeric(solve_data_final[c], errors="coerce").fillna(0).astype("int64")

solve_data_final["fastest_burst_10_moves_ms"] = pd.to_numeric(
    solve_data_final["fastest_burst_10_moves_ms"], errors="coerce"
).astype("Int64")  # remains NaN when fewer than 10 turns

# ---- 7) Set solve_index as the actual index ----
# solve_data_final = solve_data_final.set_index("solve_index", drop=True)

# Eliminating the outliers for both "longest_pause_ms" and "number_of_rotations"
solve_data_final = solve_data_final.loc[~((solve_data_final["longest_pause_ms"] > 3000) & (solve_data_final.index > 500))]
solve_data_final = solve_data_final.loc[~((solve_data_final["number_of_rotations"] > 25) & (solve_data_final.index > 500))]

# ---- 8) Done ----
# display(solve_data_final.head())
# print("\nRows:", len(solve_data_final), "Columns:", len(solve_data_final.columns))
# print("\nDtypes:\n", solve_data_final.dtypes)
# print(solve_data_final.corr(numeric_only = True))

# test_size = 0.2 imply that 20% of the data is used for testing and remaining 80% will be used for training

train_df, test_df = train_test_split(solve_data_final, test_size=0.2, random_state=2)

## 1) Inspiration

As a competitive Rubik's Cube speedsolver, I have always been driven to improve my ability. I took upon this mission at age 11, and I have made incredible progress since. My first ever solution took about 10 minutes with assistance. Now, however, my average time has dropped to about 7.8 seconds, with a record time of 3.72 seconds. My Rubik's Cube journey has led me to compete in five states for a total of 30 competitions. I am currently ranked 7th fastest in my home state of Michigan, and I hope I can continue to climb the ranks.

In this project, I selected a large sample of my virtual Rubik's Cube solutions. These solutions were all digitally completed on cstimer.net, where each key on the keyboard corresponds to a different turn or rotation on the Rubik's Cube. The website tracks and stores a number of variables for each solution, making this project possible. This dataset was compiled over the course of five years, beginning from when I discovered this feature of the website in December 2020.

Through this analysis, I hope to gain valuable information about the factors that are the strongest influences on my solving speed. Perhaps the data may shine a light on what I should emphasize during my solves, contributing to my ultimate goal of a reduction in my average solving time. I hope this analysis will reveal any interesting trends that I was not previously aware of.

## 2) Stakeholders

The primary stakeholder of this project is myself. Not only am I inherently interested in discovering any trends in my data, but also this may have a notable impact on the strategies I used to improve my solve time moving forward. Based on the conclusions of this analysis, I may shift focus during my practice sessions to better address the factors that most greatly affect the speed of the solution. Perhaps I will emphasize increasing my turning speed, prioritizing efficiency, limiting rotations, reducing pauses, or make adjustments with a number of other contributing variables.

As I chase my first sub-8 second average in competition, any improvement whatsoever could play a major role in my results.

In addition, secondary stakeholders include the entire Rubik's Cube speedsolving community. Other speedsolvers will surely be interested in seeing the relationships I discover, as it may help them with their own progression journeys. I look forward to sharing the conclusions of this project with the members of the Northwestern Cubing Club and the other competitors at my Rubik's Cube tournaments.

## 3) Task and Metrics

I will implement a regression task since my response is solve time, which is a continuous numeric variable. I would like to discover how the variables in my model explain solve time, with an emphasis on the variables than I can control in my solutions.

My evaluation metrics will include both R^2 and mean absolute error (MAE). 

R^2 is a measure of how much of the variation in solve time can be explained by the variation in my predictor variables. It is a standardized metric to indicate the explanatory power of my model. 

I will also use MAE to compare between models. The greatest residuals will likely be in the first few hundred observations in the dataset, and I do not want that to account for the performance of the model too strongly. RMSE disproportionately penalizes larger residuals, whereas MAE treats residuals proportional to their sizes. Therefore, MAE is a better evaluation metric for the dataset.

## 4) Data

I exported my session data from cstimer.net using the embedded "export to file" feature in the website. The file was 29 MB in size, and consisted of all my sessions. I needed to narrow down the data to only my virtual Rubik's Cube solutions, so I only extracted "session36" from the .txt file.

Because the data was very sophisticated in nature, with the permission of Professor Behera, I used AI to write code in order to cleanly store my relevant session data in a Pandas dataframe. I instructed it to write several functions to compute values that are not directly apparent from the exported data, but are useful to my analysis.

There are a total of 22,442 observations and 15 variables. Each observation represents a different virtual Rubik's Cube solution, and each variable represents some quantitative or qualitative aspect of the solution. 

#### Numeric Variables

1. "solve_index" - The index that the solution was completed at within my session
2. "solve_time_s" - The amount of time (in seconds) taken to return the Rubik's Cube to a solved state
3. "number_of_turns" - The number of times a side of the Rubik's Cube was turned
4. "turn_speed_tps" - The number of turns completed every second (e.g. "number_of_turns" / "solve_time_s")
5. "number_of_rotations" - The number of times the user flips the cube to view another angle
6. "total_move_count" - A sum of the number of turns and the number of rotations (total keyboard clicks)
7. "longest_pause_ms" - The greatest gap in time between two consecutive moves
8. "fastest_burst_10_moves_ms" - The shortest time period in which ten consecutive turns take place
9. "wasted_moves" - The number of times a move is immediately followed by its inverse, thereby leaving the state of the cube unchanged
10. "scramble_length" - The number of turns specified in the computer-generated scramble

#### Categorical Variables

1. "date" - The date of the solution provided in (YYYY-MM-DD) format
2. "day_of_week" - The day of week of the solution
3. "time" - The time of day of the solution given in military time
4. "time_of_day_bucket" - One of four labels based on the time of day of the solution (Late Night, Morning, Afternoon, Evening)
5. "scramble" - The computer-generated sequence of turns, expressed in speedcubing notation, that reaches the cube's initial state
   
Of the 15 variables, ten are numeric and five are categorical. Four of the categorical variables are derived from the timestamp assigned to each solution, while the other is the computer-generated scramble. Though the scramble is unlikely to yield any significant conclusions in my analysis, it will be helpful to me when reconstructing solutions manually and hence it would be unwise to discard that information.

By nature of the dataset, there are no missing values. There have been occasions throughout the past five years where I have hit the "esc" key before finishing the solution, but I always delete those solutions from the session so that they would not interfere with my rolling averages. In a way, I have already manually cleaned the data through its entire existence. A number of outlier solves have been removed if there is sufficient evidence that the solution was not a genuine representation of my best effort.

The response variable is "solve_time_s", which measures how much time (in seconds) the solution took me to complete. All other variables are potential predictors, except for "scramble" since that is neither numeric nor categorical.

## 5) Prediction

In [53]:
#| echo: false
model_df = pd.DataFrame(columns=["Model Number", "New Predictor Variable(s)", "Training R^2", "Test R^2", "Training MAE", "Test MAE"])

### 5a) Linear and Unregularized Model

I will add predictors to the model in an order that I think makes most sense to me as a speedcuber. I will not include variables that I have limited control over, such as number of turns, which varies between solutions mostly due to random chance. I would like for the relationships found in the analysis to have a concrete impact on my solving, so I need predictors that I have a large degree of influence on, such as turning speed and longest pause.

To begin, the first predictor variable I will use is "solve_index". This is arguably the most important variable for the purpose of this analysis because it is the only one capable of showing my progression over time. I would expect the solve index to explain my solve time very generally, as it does not account for the variation for solutions that are close together in index.

In [54]:
#| echo: false
X_train1 = train_df[['solve_index']]
y_train1 = train_df['solve_time_s']

X_test1 = test_df[['solve_index']]
y_test1 = test_df['solve_time_s']

model1 = LinearRegression()
model1.fit(X_train1, y_train1)

train1_pred = model1.predict(X_train1)
test1_pred = model1.predict(X_test1)

train1_R2 = r2_score(y_train1, train1_pred)
test1_R2 = r2_score(y_test1, test1_pred)

train1_mae = mean_absolute_error(y_train1, train1_pred)
test1_mae = mean_absolute_error(y_test1, test1_pred)

model_df.loc[0] = [1, "solve_index", train1_R2, test1_R2, train1_mae, test1_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589


The R^2 for just one predictor remains relatively low in both the training and test data. Interestly enough, the R^2 value is higher in the test data. The MAE is between 2.1-2.2 seconds for the test data, which means that the model is, on average, off by about 2.15 seconds. That is a satisfactory start given just one variable is being used at the predictor.

We will now add "turn_speed_tps" and "longest_pause_ms" as predictors, in that order.

In [55]:
#| echo: false
X_train2 = train_df[['solve_index', 'turn_speed_tps']]
y_train2 = train_df['solve_time_s']

X_test2 = test_df[['solve_index', 'turn_speed_tps']]
y_test2 = test_df['solve_time_s']

model2 = LinearRegression()
model2.fit(X_train2, y_train2)

train2_pred = model2.predict(X_train2)
test2_pred = model2.predict(X_test2)

train2_R2 = r2_score(y_train2, train2_pred)
test2_R2 = r2_score(y_test2, test2_pred)

train2_mae = mean_absolute_error(y_train2, train2_pred)
test2_mae = mean_absolute_error(y_test2, test2_pred)

model_df.loc[1] = [2, "turn_speed_tps", train2_R2, test2_R2, train2_mae, test2_mae]

In [56]:
#| echo: false
X_train3 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms']]
y_train3 = train_df['solve_time_s']

X_test3 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms']]
y_test3 = test_df['solve_time_s']

model3 = LinearRegression()
model3.fit(X_train3, y_train3)

train3_pred = model3.predict(X_train3)
test3_pred = model3.predict(X_test3)

train3_R2 = r2_score(y_train3, train3_pred)
test3_R2 = r2_score(y_test3, test3_pred)

train3_mae = mean_absolute_error(y_train3, train3_pred)
test3_mae = mean_absolute_error(y_test3, test3_pred)

model_df.loc[2] = [3, "longest_pause_ms", train3_R2, test3_R2, train3_mae, test3_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.817574
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408


When both variables were added, the R^2 increased substantially for both the training and test data. This is because these variables allow the model to differentiate between solutions that are close together in index. Individual variation between solves is accounted for by this model. While the MAE did decline after adding "turn_speed_tps", meaning our errors are now smaller, it did rise back up again after adding "longest_pause_ms". The reason is not entirely clear, but it may have something to do with the model trying to overexplain outlier solutions with the new information given by the "longest_pause_ms" predictor variable.

Next, we will add "number_of_rotations", "fastest_burst_10_moves_ms", and "wasted_moves".

In [57]:
#| echo: false
X_train4 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations']]
y_train4 = train_df['solve_time_s']

X_test4 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations']]
y_test4 = test_df['solve_time_s']

model4 = LinearRegression()
model4.fit(X_train4, y_train4)

train4_pred = model4.predict(X_train4)
test4_pred = model4.predict(X_test4)

train4_R2 = r2_score(y_train4, train4_pred)
test4_R2 = r2_score(y_test4, test4_pred)

train4_mae = mean_absolute_error(y_train4, train4_pred)
test4_mae = mean_absolute_error(y_test4, test4_pred)

model_df.loc[3] = [4, "number_of_rotations", train4_R2, test4_R2, train4_mae, test4_mae]

In [58]:
#| echo: false
X_train5 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms']]
y_train5 = train_df['solve_time_s']

X_test5 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms']]
y_test5 = test_df['solve_time_s']

model5 = LinearRegression()
model5.fit(X_train5, y_train5)

train5_pred = model5.predict(X_train5)
test5_pred = model5.predict(X_test5)

train5_R2 = r2_score(y_train5, train5_pred)
test5_R2 = r2_score(y_test5, test5_pred)

train5_mae = mean_absolute_error(y_train5, train5_pred)
test5_mae = mean_absolute_error(y_test5, test5_pred)

model_df.loc[4] = [5, "fastest_burst_10_moves_ms", train5_R2, test5_R2, train5_mae, test5_mae]

In [59]:
#| echo: false
X_train6 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves']]
y_train6 = train_df['solve_time_s']

X_test6 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves']]
y_test6 = test_df['solve_time_s']

model6 = LinearRegression()
model6.fit(X_train6, y_train6)

train6_pred = model6.predict(X_train6)
test6_pred = model6.predict(X_test6)

train6_R2 = r2_score(y_train6, train6_pred)
test6_R2 = r2_score(y_test6, test6_pred)

train6_mae = mean_absolute_error(y_train6, train6_pred)
test6_mae = mean_absolute_error(y_test6, test6_pred)

model_df.loc[5] = [6, "wasted_moves", train6_R2, test6_R2, train6_mae, test6_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.817574
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248


The R^2 values continued to increase as more predictors were added, though not as fast as before. Still, the R^2 for the test data remains higher than it is for the training data. One likely explanation is that the training data contains a larger proportion of outlier data. Most of these high influence points are likely to be found in the first few hundred indices of the dataset, where the times were much higher, yet rapidly declining with high variation. The MAE also continued to improve, but only slightly better than the level it was at before when we only had two predictor variables.

Finally, we will add our last two linear terms, which are "day_of_week" and "time_of_day_bucket". Both of these are categorical, so they will need to be one-hot encoded before being used in the model.

In [60]:
#| echo: false
X_train7 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week']]
y_train7 = train_df['solve_time_s']

X_test7 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week']]
y_test7 = test_df['solve_time_s']

X_train7_ohe = pd.get_dummies(X_train7, drop_first = True)
X_test7_ohe = pd.get_dummies(X_test7, drop_first = True)

model7 = LinearRegression()
model7.fit(X_train7_ohe, y_train7)

train7_pred = model7.predict(X_train7_ohe)
test7_pred = model7.predict(X_test7_ohe)

train7_R2 = r2_score(y_train7, train7_pred)
test7_R2 = r2_score(y_test7, test7_pred)

train7_mae = mean_absolute_error(y_train7, train7_pred)
test7_mae = mean_absolute_error(y_test7, test7_pred)

model_df.loc[6] = [7, "day_of_week", train7_R2, test7_R2, train7_mae, test7_mae]

In [61]:
#| echo: false
X_train8 = train_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week', 'time_of_day_bucket']]
y_train8 = train_df['solve_time_s']

X_test8 = test_df[['solve_index', 'turn_speed_tps', 'longest_pause_ms', 'number_of_rotations', 'fastest_burst_10_moves_ms', 'wasted_moves', 'day_of_week', 'time_of_day_bucket']]
y_test8 = test_df['solve_time_s']

X_train8_ohe = pd.get_dummies(X_train8, drop_first = True)
X_test8_ohe = pd.get_dummies(X_test8, drop_first = True)

model8 = LinearRegression()
model8.fit(X_train8_ohe, y_train8)

train8_pred = model8.predict(X_train8_ohe)
test8_pred = model8.predict(X_test8_ohe)

train8_R2 = r2_score(y_train8, train8_pred)
test8_R2 = r2_score(y_test8, test8_pred)

train8_mae = mean_absolute_error(y_train8, train8_pred)
test8_mae = mean_absolute_error(y_test8, test8_pred)

model_df.loc[7] = [8, "time_of_day_bucket", train8_R2, test8_R2, train8_mae, test8_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.817574
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248
6,7,day_of_week,0.640835,0.702479,1.800899,1.758624
7,8,time_of_day_bucket,0.640908,0.702498,1.800833,1.759127


The categorical, temporal variables had little to no effect on both the R^2 and MAE values for training and test data. This shows that very little information is learned about the speed of a solution if the weekday or time of day bucket is known. The one exception is that many of the early solutions were on Tuesday in the afternoon, so those bias the impact of the categorical variables.

### 5b) Non-Linear Terms

And finally, we will add non-linear terms, which include both transformation and interaction terms. The model will also be regularized with Ridge to avoid overfitting to the training data. It will discriminate between the non-linear terms that are useful and the ones that are not.

In [62]:
#| echo: false
poly = PolynomialFeatures(degree = 2, include_bias = False)
X_train_poly = poly.fit_transform(X_train8_ohe)
X_test_poly = poly.fit_transform(X_test8_ohe)

scaler = StandardScaler()
scaler.fit(X_train_poly)
X_train_poly_scaled = scaler.transform(X_train_poly)
X_test_poly_scaled = scaler.transform(X_test_poly)

rcv = RidgeCV(cv = 5)
rcv.fit(X_train_poly_scaled, y_train8)

train_poly_pred = rcv.predict(X_train_poly_scaled)
test_poly_pred = rcv.predict(X_test_poly_scaled)

train9_R2 = r2_score(y_train8, train_poly_pred)
test9_R2 = r2_score(y_test8, test_poly_pred)

train9_mae = mean_absolute_error(y_train8, train_poly_pred)
test9_mae = mean_absolute_error(y_test8, test_poly_pred)

model_df.loc[8] = [9, "non-linear terms", train9_R2, test9_R2, train9_mae, test9_mae]
display(model_df)

,Model Number,New Predictor Variable(s),Training R^2,Test R^2,Training MAE,Test MAE
0,1,solve_index,0.100562,0.114303,2.196326,2.122589
1,2,turn_speed_tps,0.339051,0.363274,1.822237,1.817574
2,3,longest_pause_ms,0.499753,0.582068,1.948229,1.929408
3,4,number_of_rotations,0.508713,0.585678,1.967880,1.921989
4,5,fastest_burst_10_moves_ms,0.585575,0.671342,1.952847,1.908345
5,6,wasted_moves,0.640290,0.703395,1.800461,1.751248
6,7,day_of_week,0.640835,0.702479,1.800899,1.758624
7,8,time_of_day_bucket,0.640908,0.702498,1.800833,1.759127
8,9,non-linear terms,0.891035,0.834125,1.246007,1.241614


There is a sizable improvement in all metrics. For the first time, the training R^2 is higher than the test R^2. The predictor variables now explain much of the variation in solve time. The MAE decreased to about 1.24 seconds for training and test data, which means that the model is off by about 1.24 seconds on average. There is always some degree of irreducible randomness to my solutions, so it is hard for the model to be much more accurate than this.

After tuning and trained the regularized model, some terms remain important while others have coefficients that approach 0. The three most important and three least important terms after regulzarization are shown below, along with their coefficients.

In [63]:
#| echo: false
feature_names = poly.get_feature_names_out(X_train8_ohe.columns)

my_coefs = pd.DataFrame({"Feature": feature_names, "Coefficient": rcv.coef_})

ranked_coefs = my_coefs.reindex(my_coefs["Coefficient"].abs().sort_values(ascending=False).index)
# display(ranked_coefs.head(10))
# display(ranked_coefs.tail(10))

# print("Best alpha:", rcv.alpha_)

display(ranked_coefs.head(3))
display(ranked_coefs.tail(3))

,Feature,Coefficient
3,number_of_rotations,-4.779940
46,longest_pause_ms fastest_burst_10_moves_ms,3.900237
4,fastest_burst_10_moves_ms,3.562361


,Feature,Coefficient
93,day_of_week_Monday day_of_week_Thursday,0.0
92,day_of_week_Monday day_of_week_Sunday,0.0
91,day_of_week_Monday day_of_week_Saturday,0.0


### 5c) Interpretation

The three predictors that appear to be the most informative include "number_of_rotations", the interaction term between "longest_pause_ms" and "fastest_burst_10_moves_ms", and "fastest_burst_10_moves_ms". It is unclear why these terms are the most informative to the model in particular, but I can make some educated guesses. I think these variables reflect how focused I am during my solutions. Low rotations means that I am planning ahead accurately and being more methodical in my solutions. A speedy burst of 10 moves may indicate that I am really focused and I have a good feeling for the keyboard in that moment. And no long pauses indicates that I am not distracted by my surroundings. All of these factors are advantageous for quicker solutions.

The three predictors that are least informative are all interaction terms between the days of the week categories. I think it is made abundantly clear by the model that the day of the week has almost no measurable bearing on my performance. They did not add much explanatory power at all when added to the earlier models, and even their non-linear terms have coefficients that are regularized to almost 0 by the tuned model.

Many of the highest coefficient terms in the tuned model are transformation or interaction terms, so there is sufficient evidence to suggest that there are non-linearities in the underlying relationship between the predictors and solve time. The second most informative variable is a transformation term, and six out of ten of the highest coefficients in the regularized model belong to either a transformation or interaction term. Also, the R^2 and MAE metrics both substantially improved when non-linear terms were introduced, so it is very likely that non-linear terms play a major role in the underlying relationship.

## 6) Inference

In [64]:
#| echo: false
inf_model = smf.ols("solve_time_s ~ solve_index + time_of_day_bucket + longest_pause_ms*fastest_burst_10_moves_ms + longest_pause_ms*number_of_rotations + fastest_burst_10_moves_ms*day_of_week + I(fastest_burst_10_moves_ms ** 2)", data = train_df).fit()

inf_model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           solve_time_s   R-squared:                       0.777
Model:                            OLS   Adj. R-squared:                  0.777
Method:                 Least Squares   F-statistic:                     2837.
Date:                Mon, 16 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:53:52   Log-Likelihood:                -43496.
No. Observations:               17953   AIC:                         8.704e+04
Df Residuals:                   17930   BIC:                         8.722e+04
Df Model:                          22                                         
Covariance Type:            nonrobust                                         
======================================================================================================================
                                                         coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------------------------
Intercept                                             13.2732      0.420     31.604      0.000      12.450      14.096
time_of_day_bucket[T.Evening]                         -0.0317      0.056     -0.562      0.574      -0.142       0.079
time_of_day_bucket[T.Late Night]                      -0.0820      0.060     -1.368      0.171      -0.200       0.035
time_of_day_bucket[T.Morning]                         -0.0110      0.071     -0.155      0.877      -0.151       0.129
day_of_week[T.Monday]                                 -0.4204      0.461     -0.912      0.362      -1.324       0.483
day_of_week[T.Saturday]                                0.0094      0.569      0.016      0.987      -1.106       1.125
day_of_week[T.Sunday]                                 -1.6917      0.547     -3.091      0.002      -2.764      -0.619
day_of_week[T.Thursday]                               -0.0188      0.440     -0.043      0.966      -0.882       0.844
day_of_week[T.Tuesday]                                -6.4872      0.458    -14.170      0.000      -7.385      -5.590
day_of_week[T.Wednesday]                              -1.9541      0.434     -4.503      0.000      -2.805      -1.103
solve_index                                        -9.296e-05   4.11e-06    -22.632      0.000      -0.000   -8.49e-05
longest_pause_ms                                      -0.0061      0.000    -34.143      0.000      -0.006      -0.006
fastest_burst_10_moves_ms                              0.0018      0.000      3.759      0.000       0.001       0.003
fastest_burst_10_moves_ms:day_of_week[T.Monday]        0.0006      0.001      1.083      0.279      -0.000       0.002
fastest_burst_10_moves_ms:day_of_week[T.Saturday]      0.0001      0.001      0.173      0.862      -0.001       0.001
fastest_burst_10_moves_ms:day_of_week[T.Sunday]        0.0019      0.001      3.228      0.001       0.001       0.003
fastest_burst_10_moves_ms:day_of_week[T.Thursday]   6.342e-05      0.000      0.130      0.896      -0.001       0.001
fastest_burst_10_moves_ms:day_of_week[T.Tuesday]       0.0075      0.001     14.805      0.000       0.007       0.009
fastest_burst_10_moves_ms:day_of_week[T.Wednesday]     0.0024      0.000      4.999      0.000       0.001       0.003
longest_pause_ms:fastest_burst_10_moves_ms          4.262e-06   1.54e-07     27.598      0.000    3.96e-06    4.56e-06
number_of_rotations                                   -0.2900      0.008    -34.457      0.000      -0.306      -0.273
longest_pause_ms:number_of_rotations                   0.0005   6.54e-06     79.414      0.000       0.001       0.001
I(fastest_burst_10_moves_ms ** 2)                  -1.029e-06    1.6e-07     -6.418      0.000   -1.34e-06  

**Intercept**

The intercept indicates that when the categories are the baselines (afternoon and Friday), and every other predictor is set at 0, the model predicts that the solve time is 13.2732 seconds.

**Shift Dummies for Time of Day and Day of Week**

The shift dummies assume that all else being equal, the intercept of the model is equal to 13.2732 plus the coefficient of the shift. The coefficient shows the difference between the baseline category (which is afternoon for time of day and Friday for day of week) and the category of the observation.

**Solve Index**

For a 1-unit increase in solve index, all else being equal, the model predicts a very modest -9.296e-05	change in solve time. Though this may seem incredibly small, this small improvement over time compounds. The p-value is rounded to 0.000, which means that the systematic decrease in solve time as in the index increases can not be attributed to random chance alone.

**Longest Pause**

For a 1 millisecond increase in the longest pause, the solve time is expected to drop by 0.0061 seconds. This is the opposite direction than I had expected, as a longer pause should indicate a higher solve time. This effect may be captured in the other variables, however. Interaction terms may be the source of this issue, as it means we cannot truly estimate the impact of a pure 1 millisecond increase in the longest pause.

**Fastest Burst**

For a 1  millisecond increase in the fastest burst of 10 moves, the solve time is expected to increase by 0.0018 seconds. This, on the other hand, does make logical sense. A burst that takes longer should lead to a solution that takes longer. This effect is for the baseline category of the day of the week variable, which means that this shows the effect of the fastest burst on Fridays. 

**Fastest Burst Interaction Term with Day of Week**

These values indicate how fastest burst affects solve time differently depending on the day of the week. The coefficients represent the change in the impact on solve time when moving from Friday to another weekday as fastest burst of 10 moves varies.

**Longest Pause Interaction Term with Fastest Burst**

For a 1 millisecond increase in longest pause, the solve time is expected to increase by -0.0061 + (4.262e-06 * fastest_burst_10_moves_ms). The interaction term goes both ways, and each predictor is dependent on the other. This is disregarding the other interaction terms these two predictors may have elsewhere.

**Number of Rotations**

Disregarding the interaction term, one extra rotation leads the model to predict a decrease of 0.29 seconds in solve time. This is the opposite direction since more rotations should lead to a slower solution that is longer in duration, but the interaction term is very likely behind this.

**Longest Pause Interaction Term with Number of Rotations**

For a 1 millisecond increase in longest pause, the solve time is expected to increase by -0.0061 + (0.0005 * number_of_rotations). The interaction term goes both ways, and each predictor is dependent on the other. This is disregarding the other interaction terms these two predictors may have elsewhere.

**Fastest Burst Transformation Term**

There is no clear way to interpret for a transformation term, but the coefficient is negative, meaning longer faster bursts are expected to decrease solve time.

**Reliability**

Besides for the terms that involve or include time of day or day of the week, the reliability of each predictor is very high. The p-values are extremely low for all such terms, meaning none of the effects can be explained reasonably by random chance alone. Multicollinearity may cause some added variability, but the coefficient estimates remain unbiased. The variability is rounded down to 0.000 for many of the terms, so the estimates are all more than reliable. However, the temporal variables have almost no measureable impact on solve time, so many of those terms (including non-linear ones) are statistically insignificant.

**Explained Variation by the Model**

Based on the R^2 value of the model, 77.7% of the variation in solve time can be explained by the variation in the predictor variables. This is not as high as the R^2 value with all the non-linear terms from part 5, but we selected the most important ones and still remain with a high R^2 value.

## 7) Recommendation to Stakeholders

Based on the analysis, a number of variables impact my solve time, whether they are linear or non-linear. I should deliberately practice my solving style to improve in the metrics that matter, such as working to reduce the number of rotations per solve, quickening my shortest burst of 10 moves, and limiting pausing. And even though the effect of solve index is incredibly small, it is still negative, meaning that I am improving with more solutions under my belt. It is unclear how I am supposed to address the non-linear predictors of my model, but I can only control so much. Other members of the speedcubing community can implement the same principles to their practice regimen in order to reduce their solve times.

One crucial limitation of this analysis is the medium of every solution in the dataset. All solutions were completely virtually on cstimer.net, but I care much more about my solving ability with a physical Rubik's Cube. Officially sanctioned competitions only include physical Rubik's Cube events, so if I want to climb the rankings, I would like to analyze data that can translate over to non-virtual solutions. 

Though there is significant overlap between the physical and virtual Rubik's Cube, there are just as many key distinctions. For instance, the virtual cube is much harder to view since the computer presents an angle that shows only the front and top faces, but a solver can see three faces at a time on a physical cube. Most importantly, though, the virtual Rubik's Cube has no regard for turning ergonomics. Half the battle on a Rubik's Cube is not what turns the solver is doing, but how they are doing them. I spend a lot of time optimizing my solving style to minimize regripping and overworking a finger. For instance, I may complete a turn of the top face by completing a left index drag rather than a right thumb and index pinch, and this all depends on the previous positions of my hands as well as where I would like them to be at the beginning of the next sequence of moves. The virtual cube, on the other hand, is strictly completed through a keyboard.

The only way to fully overcome these limitations is by using a sample of physical Rubik's Cube solutions instead. The virtual cube is remarkably convenient because it stores each move as well as the corresponding timestamp, down to the millisecond. In order to do the same with a physical Rubik's Cube, a solver would need to use a "smart" cube. These contain internal sensors that track the moves and connect to an app via Bluetooth. Unfortunately, these cubes also tend to be heavy and suboptimal for speedcubing. A speedcuber could also conduct a manual frame-by-frame analysis on each of their solutions, but that would be immensely laborious.

Rather than overcoming the disconnect between virtual and physical solutions, it is best to recognize the limited scope of the analysis. There are enough commonalities that general trends and insights will remain the same, but the virtual Rubik's Cube can not be used as a perfect proxy for physical Rubik's Cube speedsolving.

## 8) Conclusion

Taken together, the analysis reveal a holistic understanding of my speedcubing performance that cannot be captured with one model alone. A number of variables contribute to every solution I complete, whether it is my control or not. As a speedcuber on a mission to progress as much as possible, I need to be more intentional in limiting rotations, reducing pauses, reaching a higher top-end turn speed, and eliminating wasted moves. Non-linear terms also play a major role, but that is difficult to implement in a training regiment.

No one metric alone will ever encompass my speedcubing performance. I heard a speedcuber offer an analogy before, and I believe sharing it would be pertinent. A speedcuber can be compared to a ship at sea. When they first start off, the ship is riddled with holes which lead to it beginning to sink. We are only humans, so we can only patch one hole at a time. For a speedcuber, they may "patch their hole" by learning a new algorithm or discovering another angle to recognize a pattern. Everyone begins with a number of weaknesses. However, after they patch their holes enough times, one might realize that their ship is staying afloat quite well. There are too many facets of speedcubing to focus on at any one moment, but in the long run, progression does happen when the weakest areas are targeted and eliminated.

This concept of speedcubing progression extends far beyond the edges of the 3x3x3 puzzle. Being deliberate with practice sessions, addressing weaknesses, and persisting even when results are not readily apparent are all skills that are applicable to any field of human endeavor. Moving forward, I hope that myself and others can internalize these lessons and incorporate them into many other aspects of life.

## References

[1] cstimer.net. "Session Export Function." Accessed March 2026.
https://cstimer.net

## Appendix

The image below shows the keyboard controls for the virtual Rubik's Cube, expressed in speedcubing notation.

<img src="virtualcubekeymap.png" width="400">

To try the virtual Rubik's Cube for yourself, open cstimer.net using the link in the references section above.

1. Click on "option" with the gear icon in the top left corner of the screen.
2. Select "timer" on the menu along the left-hand side.
3. For the row "entering in times with: ", use the dropdown menu to change from "timer" to "virtual".
4. Exit by pressing "OK" toward the bottom.

To start, simply press the space bar. The timer will begin upon the first turn. The timer will stop automatically once the Rubik's Cube has been solved or the user presses the "esc" key.